In [1]:
import pennylane as qml
import numpy as np

def U(x):
    """U function that creates the gate that checks the parity of the number of forests.

    The input state (8 bits) is loaded into wires 0-7; the ancilla/output is wire 8.
    The function should not return anything; it simply applies gates.
    """
    # Prepare input state: set qubits according to input bitstring
    for i in range(8):
        if x[i]:
            qml.PauliX(wires=i)

    # Forest parity computation: wire 8 flips for each forest start
    # First bit: if qubit 0 == 1, it's a forest start -> flip ancilla wire 8
    qml.CNOT(wires=[0,8])
    # For i=1..7: forest start if wire[i-1]==0 and wire[i]==1
    for i in range(1,8):
        qml.PauliX(wires=i-1)    # Flip previous qubit so control is on 0
        qml.Toffoli(wires=[i-1, i, 8])   # Toffoli([prev,i], target=ancilla)
        qml.PauliX(wires=i-1)    # Restore previous qubit

def count_forest_parity(x):
    """Returns the parity of forests (even: 0, odd: 1) for given input x using quantum circuit."""
    dev = qml.device("default.qubit", wires=9)
    @qml.qnode(dev)
    def circuit():
        U(x)
        return qml.expval(qml.PauliZ(8))
    result = circuit()
    # Convert expectation value to 0 (even) or 1 (odd)
    # |0>: +1, |1>: -1 -- (1 - value) / 2
    return int(np.round((1-result)/2))

# Test cases
def test():
    # [1,0,1,1,0,1,1,1] -> 3 forests -> odd -> 1
    assert count_forest_parity([1,0,1,1,0,1,1,1]) == 1
    # [0,0,0,0,0,1,0,1] -> 2 forests -> even -> 0
    assert count_forest_parity([0,0,0,0,0,1,0,1]) == 0
    # All trees: [1,1,1,1,1,1,1,1] -> 1 forest -> odd -> 1
    assert count_forest_parity([1,1,1,1,1,1,1,1]) == 1
    # No forests: [0,0,0,0,0,0,0,0] -> 0 forests -> even -> 0
    assert count_forest_parity([0,0,0,0,0,0,0,0]) == 0
    # Forest at start and end: [1,0,0,0,0,0,0,1] -> 2 forests -> even -> 0
    assert count_forest_parity([1,0,0,0,0,0,0,1]) == 0
    # Alternating: [1,0,1,0,1,0,1,0] -> 4 forests -> even -> 0
    assert count_forest_parity([1,0,1,0,1,0,1,0]) == 0
    # Single tree at end: [0,0,0,0,0,0,0,1] -> 1 forest -> odd -> 1
    assert count_forest_parity([0,0,0,0,0,0,0,1]) == 1
    # Random: [1,1,0,1,0,1,0,0] -> forests at 0,2,4  (indices) -> 3 forests -> odd
    assert count_forest_parity([1,1,0,1,0,1,0,0]) == 1

test()
